# 🌊 Week 4, Day 2 — June 9, 2026
## Source Attribution Analysis



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw","processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata","visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026
import seaborn as sns

In [ ]:
master   = pd.read_csv(DIRS['processed']+'/master_table_v1.csv')
hotspots = pd.read_csv(DIRS['processed']+'/hotspots.csv')
df_world = pd.read_csv(DIRS['processed']+'/plastic_world.csv')
df_river = pd.read_csv(DIRS['processed']+'/river_india.csv')
print(f'master: {master.shape}, hotspots: {len(hotspots)}, river_india: {len(df_river)}')

## Step 1: Plastic Type Distribution in Hotspot Zones

In [ ]:
# Expand the pipe-separated Plastic_Types_Present column
plastic_type_counts = {}
for row in hotspots['Plastic_Types_Present'].dropna():
    for pt in row.split('|'):
        plastic_type_counts[pt] = plastic_type_counts.get(pt, 0) + 1

print('Plastic type prevalence in hotspot cells:')
for k, v in sorted(plastic_type_counts.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v} cells')

# Plastic type → likely source attribution
source_map = {
    'PET': 'Consumer_Goods',
    'PE':  'Consumer_Goods',
    'PP':  'Consumer_Goods',
    'PS':  'Industrial',
    'PVC': 'Industrial',
}
print()
print('Source attribution by plastic type:')
for pt, src in source_map.items():
    print(f'  {pt} → {src}')

## Step 2: River Plastic Contribution to Hotspot Zones

In [ ]:
# Which Indian rivers discharge closest to hotspot coordinates?
print('India river plastic risk (top 15 by plastic flow):')
river_cols = [c for c in df_river.columns if 'Plastic' in c or 'River' in c or 'Country' in c]
print(df_river.nlargest(15,'Plastic_to_River_2015_tons')[['River_Name','Plastic_to_River_2015_tons','Risk_Category']].to_string(index=False))

## Step 3: Hotspot Source Summary by Ocean Zone

In [ ]:
zone_summary = master.dropna(subset=['Plastic_Density_kg_km2']).groupby('Ocean_Zone').agg(
    Total_Density  = ('Plastic_Density_kg_km2','sum'),
    Avg_Density    = ('Plastic_Density_kg_km2','mean'),
    Cell_Count     = ('grid_lat','count'),
).round(4).reset_index()

zone_summary['Regional_Contribution_Pct'] = (
    zone_summary['Total_Density'] / zone_summary['Total_Density'].sum() * 100
).round(1)

print('Regional Contribution to Total Plastic Density:')
print(zone_summary.to_string(index=False))
print()
print('India total plastic waste: 26.33 MT | Main source: Consumer_Goods | Coastal Risk: HIGH')
print('Recycling rate: 11.5% — one of the lowest among top polluters')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Zone contribution pie
axes[0].pie(zone_summary['Regional_Contribution_Pct'],
            labels=zone_summary['Ocean_Zone'],
            autopct='%1.1f%%', startangle=90)
axes[0].set_title('Regional Contribution to\nPlastic Density', fontweight='bold')

# Plastic type bar
types = list(plastic_type_counts.keys())
counts = [plastic_type_counts[t] for t in types]
axes[1].bar(types, counts, color=['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6'])
axes[1].set_title('Plastic Types in Hotspot Zones', fontweight='bold')
axes[1].set_xlabel('Plastic Type')
axes[1].set_ylabel('Grid Cells with Presence')
axes[1].grid(True, alpha=0.3)

plt.suptitle('June 9, 2026 — Source Attribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIRS['visualizations']+'/Week4_Day2_source_attribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Source attribution chart saved ✅')